# 四方案对抗自对弈实验执行指南

## 基础修复 (已集成到所有方案)

v1 存在**致命缺陷**: Challenger 奖励函数的对抗信号 `adversarial_bonus = 0.25 × verifier_asr` 是**类别级常量**，
在 GRPO 的 `advantage = reward - group_mean` 中被消去 → Challenger 从未学到如何骗过 Reviewer。

**修复**: 将对抗信号改为**逐样本** `reviewer_fooled`，`R = quality_gate × (fooled ? 1.0 : 0.0)`。
此修复已集成到以下全部 4 个方案中，不作为独立方案。

## 方案总览

| # | 方案 | 核心思路 | 独立改动位置 | 复杂度 |
|---|------|----------|------------|--------|
| **基线** | **v1 + 奖励修复** | 仅修复对抗信号 (对照组) | Phase 0 + Phase A | ★ |
| **Plan 2** | **REINFORCE** | num_generations=1 退化为 REINFORCE | 仅超参 | ★ |
| **Plan 3** | **Data Pipeline** | 拒绝采样 + 自适应回放 + 多轮缓冲 | Phase 0 + Phase B | ★★ |
| **Plan 4** | **API Verifier** | 用 72B+ API 模型替代 7B Verifier | Phase 0 | ★★ |

所有方案均包含基础奖励修复。基线方案用于控制变量对照。

---

## 各方案详细原理

### 基线: v1 + 奖励修复 (对照组)

除奖励修复外，其他超参和数据管线与 v1 完全相同。
这是其他 3 个方案的**对照基准**: 通过比较可以隔离出每个 Plan 独有改动的贡献。

### Plan 2: REINFORCE (单采样策略梯度)

**解决什么问题？**  
GRPO 的 `num_generations=4` 意味着每个 prompt 要采样 4 次再对比。  
当 4 个采样都获得相似 reward (如都成功或都失败)，advantage ≈ 0 → 浪费计算。  
SSP 论文 (arxiv 2510.18821) 中 Proposer 端实际用 REINFORCE (n=1)。

**怎么改？**  
- `C_NUM_GEN=1`: TRL GRPOTrainer 退化为 REINFORCE (advantage = reward - running_mean)
- `MAX_STEPS=200`: 4 倍步数补偿采样效率降低
- **推理量降低 75%，每轮 Phase A 更快**
- 在基础奖励修复之上，仅改两个超参

### Plan 3: Data Pipeline 优化

**解决什么问题？**  
1. **垃圾样本污染**: Challenger 生成的 AI 拒绝 / 重复文本 / 太短文本占据 batch 且 reward≈-1
2. **种子数据比例固定**: v1 `seed_ratio=2.0` 在后期阶段给太多种子数据，限制对新模式的适应
3. **无历史数据复用**: 每轮只用本轮数据+种子，丢弃前几轮的有价值样本

**三项子改进：**
- **3a 拒绝采样**: Phase 0 Step 3.5 用 quality_gate 过滤低质量生成 (threshold=0.3)
- **3b 自适应回放**: seed_ratio 随轮次递减 (3.0→2.0→1.0→0.5)  
- **3c 多轮缓冲区**: Reviewer SFT 混合最近 K=2 轮的动态数据 (SSP 论文 Periodic Reset)

### Plan 4: API Verifier (外部大模型验证器)

**解决什么问题？**  
v1 的 Verifier 是**冻结的 7B LoRA Reviewer**。7B 模型的分类准确率有限 (~63%)，  
可能给出错误的 ground truth → 奖励信号噪声大。  
而且 7B Verifier 和训练中的 Reviewer 同架构 → 共享判断偏见。

**怎么改？**  
- 用 72B+ API 模型 (如 qwen-plus) 替代本地 7B Verifier
- **更准确的标签 + 架构去偏**
- Phase 0 无需加载 7B 到 NPU → 显存全留给 Challenger + Reviewer 推理
- API 失败时自动降级到本地 7B (容错)
- 成本: ~6 元 / 5 轮 (qwen-plus)

---
## 0. 环境检查 & 路径配置

In [ ]:
import os, subprocess, json

# ═══════════════════════════════════════════════════════════
# ★★★ 请根据实际情况修改以下路径 ★★★
# ═══════════════════════════════════════════════════════════

# 训练数据和模型所在目录 (与 v1 一致)
BASE_DIR = "/home/ma-user/work/test"

# 项目代码根目录
PROJECT_DIR = "/home/ma-user/work/test/chineseharm_adversarial_training"

# Python 解释器
PYTHON_EXEC = "/home/ma-user/.conda/envs/ssp_train/bin/python"

# 训练配置
MODEL_SIZE = "3B"    # 可选: 0.5B / 1.5B / 3B / 7B / 14B
N_GPUS = 4           # NPU 卡数
SELFPLAY_ROUNDS = 5  # 自对弈轮次

# Plan 4 专用 (API Verifier)
VERIFIER_API_KEY = ""  # ← 填入你的 API Key，留空则 Plan 4 自动降级到本地 7B

# ═══════════════════════════════════════════════════════════
SCRIPTS_DIR = os.path.join(PROJECT_DIR, "scripts")
V1_DIR = os.path.join(SCRIPTS_DIR, "rl_train")

print(f"BASE_DIR:    {BASE_DIR}")
print(f"PROJECT_DIR: {PROJECT_DIR}")
print(f"SCRIPTS_DIR: {SCRIPTS_DIR}")
print(f"MODEL_SIZE:  {MODEL_SIZE}")
print(f"N_GPUS:      {N_GPUS}")

In [ ]:
# 环境检查: NPU 是否可用
!{PYTHON_EXEC} -c "
import torch
import torch_npu
n = torch.npu.device_count()
print(f'NPU 可用数量: {n}')
for i in range(n):
    props = torch.npu.get_device_properties(i)
    print(f'  NPU {i}: {props.name}')
assert n >= {N_GPUS}, f'需要 {N_GPUS} 张 NPU，但只检测到 {n} 张'
print('✅ NPU 环境正常')
"

In [ ]:
# 检查关键路径是否存在
paths_to_check = {
    "种子数据":       f"{BASE_DIR}/prepared_data/rl/train_seed.parquet",
    f"Challenger {MODEL_SIZE}": f"{BASE_DIR}/merged_models_toxicn/challenger_{MODEL_SIZE}",
    f"Reviewer {MODEL_SIZE}":   f"{BASE_DIR}/merged_models_toxicn/reviewer_{MODEL_SIZE}",
    "Verifier 7B":   f"{BASE_DIR}/merged_models_toxicn/reviewer_7B",
    "v1 scripts":    V1_DIR,
    "ds_zero2.json": f"{V1_DIR}/ds_zero2.json",
    "Plan 1 脚本":   f"{SCRIPTS_DIR}/plan_reward_shaping/run_selfplay_plan1.sh",
    "Plan 2 脚本":   f"{SCRIPTS_DIR}/plan_reinforce/run_selfplay_plan2.sh",
    "Plan 3 脚本":   f"{SCRIPTS_DIR}/plan_data_pipeline/run_selfplay_plan3.sh",
    "Plan 4 脚本":   f"{SCRIPTS_DIR}/plan_verifier_api/run_selfplay_plan4.sh",
}

all_ok = True
for name, path in paths_to_check.items():
    exists = os.path.exists(path)
    status = "✅" if exists else "❌"
    print(f"  {status} {name}: {path}")
    if not exists:
        all_ok = False

if all_ok:
    print("\n✅ 所有路径检查通过，可以开始执行")
else:
    print("\n❌ 有路径缺失，请检查后再继续")
    print("   如果 Plan 脚本缺失，需要先将 plan_* 文件夹上传到服务器")

In [ ]:
# 如果 Plan 文件夹不在服务器上，先检查并赋予执行权限
!chmod +x {SCRIPTS_DIR}/plan_reward_shaping/run_selfplay_plan1.sh
!chmod +x {SCRIPTS_DIR}/plan_reinforce/run_selfplay_plan2.sh
!chmod +x {SCRIPTS_DIR}/plan_data_pipeline/run_selfplay_plan3.sh
!chmod +x {SCRIPTS_DIR}/plan_verifier_api/run_selfplay_plan4.sh
print("✅ 脚本权限设置完成")

---
## 1. Plan 2: REINFORCE (推荐首先执行)

**为什么先跑 Plan 2？** 它零代码改动，仅改超参，最稳定。可作为 baseline 对比。

### 改动说明
```
v1:     num_generations=4 (GRPO, 每 prompt 采样 4 次)  + max_steps=50
Plan 2: num_generations=1 (REINFORCE, 每 prompt 采样 1 次) + max_steps=200
```

- 所有 Phase 0 / Phase A / Phase B 的 **Python 代码完全复用 v1**
- 只有 shell 脚本中两个超参不同
- 推理量降低 75%，但步数增加 4 倍，总训练时间相当

### 数据流
```
Phase 0: v1/generate_dynamic_data.py         → 生成 parquet
Phase A: v1/adversarial_trl_grpo.py (n=1)    → Challenger REINFORCE
Phase B: v1/mix_replay + convert_grpo_to_sft + train_reviewer_lora → Reviewer SFT
```

In [ ]:
# ═══════════════════════════════════════════════════════════
# Plan 2: REINFORCE (n=1) — 完全复用 v1，仅改超参
# ═══════════════════════════════════════════════════════════
# 
# 输出目录: {BASE_DIR}/selfplay_plan2_reinforce/{MODEL_SIZE}_{N_GPUS}npu/
# 日志目录: {BASE_DIR}/logs/plan2_reinforce_{MODEL_SIZE}/
#
# 预计时间: 与 v1 相当 (~数小时，取决于模型尺寸和 NPU 性能)
# ═══════════════════════════════════════════════════════════

plan2_cmd = f"""
cd {SCRIPTS_DIR}/plan_reinforce && \
BASE_DIR={BASE_DIR} \
MODEL_SIZE={MODEL_SIZE} \
N_GPUS={N_GPUS} \
SELFPLAY_ROUNDS={SELFPLAY_ROUNDS} \
PYTHON_EXEC={PYTHON_EXEC} \
bash run_selfplay_plan2.sh
""".strip()

print("即将执行的命令:")
print("─" * 60)
print(plan2_cmd)
print("─" * 60)
print(f"\n输出目录: {BASE_DIR}/selfplay_plan2_reinforce/{MODEL_SIZE}_{N_GPUS}npu/")
print(f"日志目录: {BASE_DIR}/logs/plan2_reinforce_{MODEL_SIZE}/")

In [ ]:
# 🚀 执行 Plan 2 (取消注释下一行即可运行)
# 注意: 这会运行很长时间，建议在终端中执行以便看到实时输出

# !{plan2_cmd}

In [ ]:
# 或者: 在终端中执行 (推荐，可实时查看输出)
# 打开一个终端 (Terminal) 窗口，粘贴以下命令:

terminal_cmd_plan2 = f"""
# Plan 2: REINFORCE — 在终端中执行
mkdir -p {BASE_DIR}/logs/plan2_reinforce_{MODEL_SIZE}
cd {SCRIPTS_DIR}/plan_reinforce
nohup bash -c '
export BASE_DIR={BASE_DIR}
export MODEL_SIZE={MODEL_SIZE}
export N_GPUS={N_GPUS}
export SELFPLAY_ROUNDS={SELFPLAY_ROUNDS}
export PYTHON_EXEC={PYTHON_EXEC}
bash run_selfplay_plan2.sh
' > {BASE_DIR}/logs/plan2_reinforce_{MODEL_SIZE}/plan2_full.log 2>&1 &
echo "Plan 2 已在后台启动，PID: $!"
echo "查看日志: tail -f {BASE_DIR}/logs/plan2_reinforce_{MODEL_SIZE}/plan2_full.log"
"""
print(terminal_cmd_plan2)

---
## 2. 基线对照: v1 + 奖励修复

**为什么要跑基线？** 它是其他 3 个方案的对照组，可以隔离每个 Plan 独有改动的贡献。

### 改动说明
```
v1:   extra_info.verifier_asr = 类别级 float (同类别样本相同) → GRPO advantage 消去 → 无梯度
修复: extra_info.reviewer_fooled = 逐样本 bool → GRPO advantage 产生真实梯度
```
其他超参和数据管线与 v1 完全相同。

### 数据流
```
Phase 0: rl_train/generate_dynamic_data_fixed.py  → 注入逐样本 reviewer_fooled
Phase A: rl_train/adversarial_trl_grpo_fixed.py   → 使用新 reward (样本级)
Phase B: v1 (mix_replay + convert_grpo_to_sft + train_reviewer_lora)
```

In [ ]:
# ═══════════════════════════════════════════════════════════
# Plan 1: Reward Shaping — 真对抗奖励信号
# ═══════════════════════════════════════════════════════════
# 基线对照: v1 + 奖励修复 (对照组)
#
# 输出目录: {BASE_DIR}/selfplay_plan1_reward_shaping/{MODEL_SIZE}_{N_GPUS}npu/
# 日志目录: {BASE_DIR}/logs/plan1_reward_shaping_{MODEL_SIZE}/
#
# 自定义 Python 文件:
#   Phase 0: generate_dynamic_data_plan1.py (注入逐样本 reviewer_fooled)
#   Phase A: adversarial_trl_grpo_plan1.py  (新 reward 函数)
#   Phase B: 完全复用 v1
# ═══════════════════════════════════════════════════════════

plan1_cmd = f"""
cd {SCRIPTS_DIR}/plan_reward_shaping && \
BASE_DIR={BASE_DIR} \
MODEL_SIZE={MODEL_SIZE} \
N_GPUS={N_GPUS} \
SELFPLAY_ROUNDS={SELFPLAY_ROUNDS} \
PYTHON_EXEC={PYTHON_EXEC} \
bash run_selfplay_plan1.sh
""".strip()

print("即将执行的命令:")
print("─" * 60)
print(plan1_cmd)
print("─" * 60)
print(f"\n输出目录: {BASE_DIR}/selfplay_plan1_reward_shaping/{MODEL_SIZE}_{N_GPUS}npu/")

In [ ]:
# 🚀 执行 Plan 1
# 取消注释下一行运行 (建议在终端中执行以查看实时日志)

# !{plan1_cmd}

In [ ]:
# 终端执行命令 (推荐)
terminal_cmd_plan1 = f"""
# Plan 1: Reward Shaping — 在终端中执行
# 基线对照: v1 + 奖励修复
mkdir -p {BASE_DIR}/logs/plan1_reward_shaping_{MODEL_SIZE}
cd {SCRIPTS_DIR}/plan_reward_shaping
nohup bash -c '
export BASE_DIR={BASE_DIR}
export MODEL_SIZE={MODEL_SIZE}
export N_GPUS={N_GPUS}
export SELFPLAY_ROUNDS={SELFPLAY_ROUNDS}
export PYTHON_EXEC={PYTHON_EXEC}
bash run_selfplay_plan1.sh
' > {BASE_DIR}/logs/plan1_reward_shaping_{MODEL_SIZE}/plan1_full.log 2>&1 &
echo "Plan 1 已在后台启动，PID: $!"
echo "查看日志: tail -f {BASE_DIR}/logs/plan1_reward_shaping_{MODEL_SIZE}/plan1_full.log"
"""
print(terminal_cmd_plan1)

---
## 3. Plan 3: Data Pipeline 优化

### 三项子改进

| 子项 | 改动 | 效果 |
|------|------|------|
| **3a 拒绝采样** | Phase 0 Step 3.5 过滤 quality_gate < 0.3 的生成 | 去除 AI 拒绝/重复/太短文本，GRPO batch 更干净 |
| **3b 自适应回放** | seed_ratio: 3.0→2.0→1.0→0.5 (随轮次) | 早期稳定，后期专注新数据 |
| **3c 多轮缓冲区** | Phase B 混合最近 K=2 轮动态数据 | 利用历史有价值样本，防遗忘 |

### 涉及的文件
- `generate_dynamic_data_plan3.py` — Phase 0 集成拒绝采样 
- `rejection_sampler.py` — quality_gate 过滤器
- `mix_replay_adaptive.py` — 自适应回放 + 多轮缓冲

### 数据流
```
Phase 0: plan3/generate_dynamic_data_plan3.py → 生成后过滤 → 更干净的 parquet
Phase A: v1/adversarial_trl_grpo.py            → 不变
Phase B: plan3/mix_replay_adaptive.py + v1/convert_grpo_to_sft + v1/train_reviewer_lora
         → 自适应混合 → SFT
```

In [ ]:
# ═══════════════════════════════════════════════════════════
# Plan 3: Data Pipeline 优化
# ═══════════════════════════════════════════════════════════
#
# 输出目录: {BASE_DIR}/selfplay_plan3_data_pipeline/{MODEL_SIZE}_{N_GPUS}npu/
#
# 可调参数 (环境变量):
#   REJECTION_THRESHOLD=0.3  — 拒绝采样阈值 (越高过滤越严)
#   BUFFER_K=2               — 回放缓冲区保留最近几轮
# ═══════════════════════════════════════════════════════════

plan3_cmd = f"""
cd {SCRIPTS_DIR}/plan_data_pipeline && \
BASE_DIR={BASE_DIR} \
MODEL_SIZE={MODEL_SIZE} \
N_GPUS={N_GPUS} \
SELFPLAY_ROUNDS={SELFPLAY_ROUNDS} \
PYTHON_EXEC={PYTHON_EXEC} \
REJECTION_THRESHOLD=0.3 \
BUFFER_K=2 \
bash run_selfplay_plan3.sh
""".strip()

print("即将执行的命令:")
print("─" * 60)
print(plan3_cmd)
print("─" * 60)
print(f"\n输出目录: {BASE_DIR}/selfplay_plan3_data_pipeline/{MODEL_SIZE}_{N_GPUS}npu/")

In [ ]:
# 🚀 执行 Plan 3

# !{plan3_cmd}

In [ ]:
# 终端执行命令 (推荐)
terminal_cmd_plan3 = f"""
# Plan 3: Data Pipeline — 在终端中执行
mkdir -p {BASE_DIR}/logs/plan3_data_pipeline_{MODEL_SIZE}
cd {SCRIPTS_DIR}/plan_data_pipeline
nohup bash -c '
export BASE_DIR={BASE_DIR}
export MODEL_SIZE={MODEL_SIZE}
export N_GPUS={N_GPUS}
export SELFPLAY_ROUNDS={SELFPLAY_ROUNDS}
export PYTHON_EXEC={PYTHON_EXEC}
export REJECTION_THRESHOLD=0.3
export BUFFER_K=2
bash run_selfplay_plan3.sh
' > {BASE_DIR}/logs/plan3_data_pipeline_{MODEL_SIZE}/plan3_full.log 2>&1 &
echo "Plan 3 已在后台启动，PID: $!"
echo "查看日志: tail -f {BASE_DIR}/logs/plan3_data_pipeline_{MODEL_SIZE}/plan3_full.log"
"""
print(terminal_cmd_plan3)

---
## 4. Plan 4: API Verifier

### 改动说明
```
v1:     本地冻结 7B LoRA Reviewer 作为 Verifier (需占用 NPU 显存)
Plan 4: 72B+ API 模型 (qwen-plus) 作为 Verifier (不占 NPU)
```

### 需要的额外配置
- `VERIFIER_API_KEY`: DashScope / OpenAI 兼容 API 的密钥
- `VERIFIER_API_BASE`: API 端点 (默认 DashScope)
- `VERIFIER_API_MODEL`: 模型名 (默认 qwen-plus)

### 涉及的文件
- `api_verifier.py` — AsyncOpenAI 异步验证器
- `generate_dynamic_data_plan4.py` — Phase 0 使用 API Verifier

### 数据流
```
Phase 0: plan4/generate_dynamic_data_plan4.py → API 验证 (失败自动降级 7B)
Phase A: v1/adversarial_trl_grpo.py           → 不变
Phase B: v1/mix_replay + convert_grpo_to_sft + train_reviewer_lora → 不变
```

### 成本估算
- 每轮: 256 条/类别 × 6 类别 = 1536 条 API 调用
- 5 轮总计: ~7680 次调用
- qwen-plus 价格: ~0.004 元/千 tokens → **总计约 6 元**

In [ ]:
# ═══════════════════════════════════════════════════════════
# Plan 4: API Verifier
# ═══════════════════════════════════════════════════════════
#
# ★ 需要先设置 API Key ★
# 如果不设置，Phase 0 会自动降级到本地 7B Verifier (等效于 v1)
#
# 支持的 API:
#   - 阿里云 DashScope (默认): qwen-plus / qwen-max
#   - OpenAI 兼容 API: DeepSeek / 本地 vLLM 等
# ═══════════════════════════════════════════════════════════

# ← 在最上面的配置格中填入 VERIFIER_API_KEY
api_key_status = "已设置" if VERIFIER_API_KEY else "未设置 (将降级到本地 7B)"
print(f"API Key 状态: {api_key_status}")

api_key_part = f'VERIFIER_API_KEY={VERIFIER_API_KEY} \\\n' if VERIFIER_API_KEY else ''

plan4_cmd = f"""
cd {SCRIPTS_DIR}/plan_verifier_api && \
BASE_DIR={BASE_DIR} \
MODEL_SIZE={MODEL_SIZE} \
N_GPUS={N_GPUS} \
SELFPLAY_ROUNDS={SELFPLAY_ROUNDS} \
PYTHON_EXEC={PYTHON_EXEC} \
{api_key_part}bash run_selfplay_plan4.sh
""".strip()

print("\n即将执行的命令:")
print("─" * 60)
print(plan4_cmd)
print("─" * 60)
print(f"\n输出目录: {BASE_DIR}/selfplay_plan4_verifier_api/{MODEL_SIZE}_{N_GPUS}npu/")

In [ ]:
# 🚀 执行 Plan 4
# 前置条件: pip install openai>=1.0 (用于 API 调用)

# !{PYTHON_EXEC} -m pip install "openai>=1.0" -q
# !{plan4_cmd}

In [ ]:
# 终端执行命令 (推荐)
api_env = f"export VERIFIER_API_KEY={VERIFIER_API_KEY}\n" if VERIFIER_API_KEY else "# VERIFIER_API_KEY 未设置, 将降级到本地 7B\n"
terminal_cmd_plan4 = f"""
# Plan 4: API Verifier — 在终端中执行
mkdir -p {BASE_DIR}/logs/selfplay_plan4_{MODEL_SIZE}
cd {SCRIPTS_DIR}/plan_verifier_api
nohup bash -c '
export BASE_DIR={BASE_DIR}
export MODEL_SIZE={MODEL_SIZE}
export N_GPUS={N_GPUS}
export SELFPLAY_ROUNDS={SELFPLAY_ROUNDS}
export PYTHON_EXEC={PYTHON_EXEC}
{api_env}bash run_selfplay_plan4.sh
' > {BASE_DIR}/logs/selfplay_plan4_{MODEL_SIZE}/plan4_full.log 2>&1 &
echo "Plan 4 已在后台启动，PID: $!"
echo "查看日志: tail -f {BASE_DIR}/logs/selfplay_plan4_{MODEL_SIZE}/plan4_full.log"
"""
print(terminal_cmd_plan4)

---
## 5. 进度监控 & 结果查看

In [ ]:
# 查看各方案训练进度
import json, glob

plans = {
    "Plan 1 (Reward Shaping)": f"{BASE_DIR}/selfplay_plan1_reward_shaping/{MODEL_SIZE}_{N_GPUS}npu/progress.json",
    "Plan 2 (REINFORCE)":     f"{BASE_DIR}/selfplay_plan2_reinforce/{MODEL_SIZE}_{N_GPUS}npu/progress.json",
    "Plan 3 (Data Pipeline)": f"{BASE_DIR}/selfplay_plan3_data_pipeline/{MODEL_SIZE}_{N_GPUS}npu/progress.json",
    "Plan 4 (API Verifier)":  f"{BASE_DIR}/selfplay_plan4_verifier_api/{MODEL_SIZE}_{N_GPUS}npu/progress.json",
    "v1 Baseline":            f"{BASE_DIR}/selfplay_outputs_sft_reviewer/{MODEL_SIZE}_{N_GPUS}npu/progress.json",
}

print("═" * 70)
print(f"{'方案':<25} {'已完成轮次':>10} {'最后阶段':>10} {'时间':>22}")
print("═" * 70)

for name, path in plans.items():
    if os.path.exists(path):
        with open(path) as f:
            p = json.load(f)
        rnd = p.get('last_completed_round', '?')
        total = p.get('total_rounds', '?')
        phase = p.get('last_completed_phase', '?')
        ts = p.get('timestamp', '?')[:19]
        print(f"  {name:<23} {rnd}/{total:>8} {phase:>10} {ts:>22}")
    else:
        print(f"  {name:<23} {'未开始':>10}")

print("═" * 70)

In [ ]:
# 查看各方案每轮的 ASR 变化 (对抗成功率)
import json, glob

plan_data_dirs = {
    "Plan 1": f"{BASE_DIR}/selfplay_dynamic_data_plan1/{MODEL_SIZE}",
    "Plan 2": f"{BASE_DIR}/selfplay_dynamic_data_plan2/{MODEL_SIZE}",
    "Plan 3": f"{BASE_DIR}/selfplay_dynamic_data_plan3/{MODEL_SIZE}",
    "Plan 4": f"{BASE_DIR}/selfplay_plan4_data/{MODEL_SIZE}",
    "v1":     f"{BASE_DIR}/selfplay_dynamic_data/{MODEL_SIZE}",
}

print("各方案每轮 ASR (Verifier 确认的对抗成功率):\n")
for plan_name, data_dir in plan_data_dirs.items():
    stats_files = sorted(glob.glob(f"{data_dir}/round_*/selfplay_stats_round*.json"))
    if not stats_files:
        continue
    asrs = []
    for sf in stats_files:
        with open(sf) as f:
            s = json.load(f)
        asrs.append(s.get('overall_verifier_asr', 0))
    print(f"  {plan_name}: " + " → ".join(f"R{i+1}={a:.3f}" for i, a in enumerate(asrs)))

In [ ]:
# 查看最新轮次的实时日志 (修改 PLAN 和 ROUND 变量)
PLAN = 1   # 1/2/3/4
ROUND = 1  # 轮次

log_dirs = {
    1: f"{BASE_DIR}/logs/plan1_reward_shaping_{MODEL_SIZE}",
    2: f"{BASE_DIR}/logs/plan2_reinforce_{MODEL_SIZE}",
    3: f"{BASE_DIR}/logs/plan3_data_pipeline_{MODEL_SIZE}",
    4: f"{BASE_DIR}/logs/selfplay_plan4_{MODEL_SIZE}",
}

log_dir = log_dirs[PLAN]
print(f"Plan {PLAN} 日志目录: {log_dir}")
print("\n最近日志文件:")
!ls -lt {log_dir}/ 2>/dev/null | head -10

# 查看最新日志的最后 50 行
print("\n" + "─" * 60)
!ls -t {log_dir}/ 2>/dev/null | head -1 | xargs -I{{}} tail -50 {log_dir}/{{}}

---
## 6. 训练完成后: 评估

每个方案训练完成后，用 ChineseHarm-bench 评估最终 Reviewer 模型。

In [ ]:
# 训练完成后: 评估各方案的最终模型
# 填入要评估的方案编号
EVAL_PLAN = 1  # 1/2/3/4

# 读取 progress.json 获取最终模型路径
progress_files = {
    1: f"{BASE_DIR}/selfplay_plan1_reward_shaping/{MODEL_SIZE}_{N_GPUS}npu/progress.json",
    2: f"{BASE_DIR}/selfplay_plan2_reinforce/{MODEL_SIZE}_{N_GPUS}npu/progress.json",
    3: f"{BASE_DIR}/selfplay_plan3_data_pipeline/{MODEL_SIZE}_{N_GPUS}npu/progress.json",
    4: f"{BASE_DIR}/selfplay_plan4_verifier_api/{MODEL_SIZE}_{N_GPUS}npu/progress.json",
}

pf = progress_files[EVAL_PLAN]
if os.path.exists(pf):
    with open(pf) as f:
        p = json.load(f)
    reviewer_path = p.get('current_reviewer', '')
    challenger_path = p.get('current_challenger', '')
    print(f"Plan {EVAL_PLAN} 最终模型:")
    print(f"  Challenger: {challenger_path}")
    print(f"  Reviewer:   {reviewer_path}")
    
    if reviewer_path and os.path.isdir(reviewer_path):
        eval_cmd = f"""
# 评估 Plan {EVAL_PLAN} 的最终 Reviewer
cd {SCRIPTS_DIR}/model_eval
{PYTHON_EXEC} eval_challenger_generation.py \\
    --model_path {reviewer_path} \\
    --test_data {BASE_DIR}/split_data/test.json \\
    --output_dir {BASE_DIR}/eval_results/plan{EVAL_PLAN}_{MODEL_SIZE}
"""
        print(f"\n评估命令:")
        print(eval_cmd)
    else:
        print("\n⚠️ 模型路径不存在，训练可能尚未完成")
else:
    print(f"❌ Plan {EVAL_PLAN} 尚未开始训练")

---
## 注意事项

### 资源占用
- **每个方案独占 4 张 NPU**，不能同时运行两个方案
- **请串行执行**: 等一个方案跑完再启动下一个
- 如果有多台机器，可以在不同机器上并行

- **推荐执行顺序**: 四个方案完全独立，开 4 个实例同时并行执行

### 推荐执行顺序
1. **基线** (v1 + 奖励修复) — 对照组
2. **Plan 2** (REINFORCE) — 最简单，仅改超参
3. **Plan 3** (Data Pipeline) — 数据管线优化
4. **Plan 4** (API Verifier) — 需要 API key

### 断点续训
- 所有方案都支持 **断点续训**: 如果训练中断，重新执行同样命令即可自动恢复
- 进度保存在 `progress.json` 中
- 如需从头开始: 设置 `RESUME=0` 环境变量

### 常见问题
- **OOM**: 降低 `GEN_BATCH_SIZE` 或 `C_PER_DEVICE_BS`

- **Plan 4 API 报错**: 自动降级到本地 7B，不影响训练- **Plan 2 n=1 NaN**: 确认 TRL 版本 ≥ 0.12 (`pip show trl`)